# Embeddings no corpus dos artigos STIL 2023

Este notebook treina Word2Vec, FastText e uma implementação simples de GloVe usando o corpus extraído dos artigos. Também gera nuvens de palavras com as palavras mais frequentes.

Entrada esperada: arquivo `stil2023_articles.json` ou `stil2023_articles (1).json`.

In [ ]:
%pip install -q gensim wordcloud matplotlib pandas nltk torch

In [ ]:
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import gensim.downloader as api
import pandas as pd
import torch
from gensim.models import FastText, Word2Vec
from nltk.corpus import stopwords
from wordcloud import WordCloud

nltk.download("stopwords")

## Carregar o JSON dos artigos

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    json_path = next(iter(uploaded.keys()))
except Exception:
    candidatos = [
        "stil2023_articles.json",
        "stil2023_articles (1).json",
        "output/stil2023_articles.json",
    ]
    json_path = next((p for p in candidatos if Path(p).exists()), None)
    if json_path is None:
        raise FileNotFoundError("Envie o JSON dos artigos ou coloque o arquivo no diretório do notebook.")

with open(json_path, "r", encoding="utf-8") as file:
    articles = json.load(file)

print(f"Arquivo carregado: {json_path}")
print(f"Quantidade de artigos: {len(articles)}")

## Preparar corpus textual

In [ ]:
def reparar_mojibake(texto):
    if not isinstance(texto, str):
        return ""
    try:
        return texto.encode("latin1").decode("utf-8")
    except UnicodeError:
        return texto


def texto_do_artigo(artigo):
    partes = [
        artigo.get("titulo", ""),
        artigo.get("resumo", ""),
        " ".join(artigo.get("keywords", []) or []),
        artigo.get("artigo_completo", ""),
    ]
    return reparar_mojibake(" ".join(str(p) for p in partes if p))


def tokenizar(texto):
    texto = texto.lower()
    return re.findall(r"[a-záàâãéèêíïóôõöúçñ]+", texto, flags=re.IGNORECASE)


stop_pt = set(stopwords.words("portuguese"))
stop_en = set(stopwords.words("english"))
stop_all = stop_pt | stop_en

sentencas = []
tokens_filtrados = []

for artigo in articles:
    texto = texto_do_artigo(artigo)
    tokens = tokenizar(texto)
    tokens = [t for t in tokens if len(t) > 2]
    if tokens:
        sentencas.append(tokens)
        tokens_filtrados.extend([t for t in tokens if t not in stop_all])

frequencias = Counter(tokens_filtrados)

print(f"Sentenças/documentos para treino: {len(sentencas)}")
print(f"Tokens filtrados: {len(tokens_filtrados):,}")
print(f"Vocabulário filtrado: {len(frequencias):,}")

## Nuvens de palavras das palavras mais frequentes

In [ ]:
def mostrar_nuvem(frequencias, titulo, max_words=120, nome_arquivo=None):
    wc = WordCloud(
        width=1600,
        height=800,
        background_color="white",
        colormap="viridis",
        max_words=max_words,
    ).generate_from_frequencies(dict(frequencias.most_common(max_words)))

    plt.figure(figsize=(16, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(titulo)
    plt.show()

    if nome_arquivo:
        wc.to_file(nome_arquivo)


top_palavras = pd.DataFrame(frequencias.most_common(30), columns=["palavra", "frequencia"])
display(top_palavras)

mostrar_nuvem(frequencias, "Palavras mais frequentes no corpus STIL 2023", nome_arquivo="nuvem_palavras_stil2023.png")
top_palavras.to_csv("palavras_mais_frequentes_stil2023.csv", index=False, encoding="utf-8")

## Treinar Word2Vec e FastText no corpus

In [ ]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 2
EPOCHS = 30

word2vec_model = Word2Vec(
    sentences=sentencas,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=2,
    sg=1,
    epochs=EPOCHS,
)

fasttext_model = FastText(
    sentences=sentencas,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=2,
    sg=1,
    epochs=EPOCHS,
)

print("Word2Vec vocabulário:", len(word2vec_model.wv))
print("FastText vocabulário:", len(fasttext_model.wv))

## Atividades com Word2Vec Skip-gram

Esta secao responde as atividades pedidas usando:

- Word2Vec treinado no corpus STIL 2023 com Skip-gram (`sg=1`);
- Word2Vec pre-treinado (`word2vec-google-news-300`) para comparar similaridade e operacoes vetoriais.

Observacao: o modelo pre-treinado do Google News e grande e foi treinado principalmente em ingles. Palavras em portugues podem aparecer como fora do vocabulario.

In [ ]:
def normalizar_token_pos(token):
    token = reparar_mojibake(str(token)).lower()
    encontrados = tokenizar(token)
    return encontrados[0] if encontrados else ""


def frequencias_por_pos(articles, pos_alvos):
    contador = Counter()
    for artigo in articles:
        tokens = artigo.get("artigo_tokenizado", []) or []
        tags = artigo.get("pos_tagger", []) or []
        for token, tag in zip(tokens, tags):
            token_limpo = normalizar_token_pos(token)
            if not token_limpo or len(token_limpo) <= 2:
                continue
            if token_limpo in stop_all:
                continue
            if tag in pos_alvos and token_limpo in word2vec_model.wv:
                contador[token_limpo] += 1
    return contador


substantivos = frequencias_por_pos(articles, {"NOUN", "PROPN"})
verbos = frequencias_por_pos(articles, {"VERB", "AUX"})

substantivo_mais_frequente = substantivos.most_common(1)[0][0] if substantivos else None
verbo_mais_frequente = verbos.most_common(1)[0][0] if verbos else None

termos_atividade = {
    "modelos": "modelos",
    "linguagem": "linguagem",
    "substantivo_mais_frequente": substantivo_mais_frequente,
    "verbo_mais_frequente": verbo_mais_frequente,
}

display(pd.DataFrame(
    [
        {"categoria": categoria, "termo": termo}
        for categoria, termo in termos_atividade.items()
    ]
))

In [ ]:
def vetor_word2vec_corpus(palavra):
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in word2vec_model.wv:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do Word2Vec do corpus."]})
    return pd.DataFrame({
        "dimensao": list(range(word2vec_model.vector_size)),
        "valor": word2vec_model.wv[palavra],
    })


for categoria, termo in termos_atividade.items():
    print(f"\nVetor Word2Vec corpus - {categoria}: {termo}")
    display(vetor_word2vec_corpus(termo).head(20))

In [ ]:
def similares_word2vec_corpus(palavra, topn=10):
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in word2vec_model.wv:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do Word2Vec do corpus."]})
    return pd.DataFrame(
        word2vec_model.wv.most_similar(palavra, topn=topn),
        columns=["palavra", "similaridade"],
    )


for categoria, termo in termos_atividade.items():
    print(f"\nTermos mais similares no Word2Vec do corpus - {categoria}: {termo}")
    display(similares_word2vec_corpus(termo, topn=10))

In [ ]:
CARREGAR_WORD2VEC_PRETREINADO = True
MODELO_WORD2VEC_PRETREINADO = "word2vec-google-news-300"

word2vec_pretreinado = None
if CARREGAR_WORD2VEC_PRETREINADO:
    print(f"Carregando modelo pre-treinado: {MODELO_WORD2VEC_PRETREINADO}")
    word2vec_pretreinado = api.load(MODELO_WORD2VEC_PRETREINADO)
    print(f"Vocabulario pre-treinado: {len(word2vec_pretreinado.index_to_key):,}")

In [ ]:
def similares_word2vec_pretreinado(palavra, topn=10):
    if word2vec_pretreinado is None:
        return pd.DataFrame({"aviso": ["Modelo pre-treinado nao carregado."]})
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in word2vec_pretreinado:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do Word2Vec pre-treinado."]})
    return pd.DataFrame(
        word2vec_pretreinado.most_similar(palavra, topn=topn),
        columns=["palavra", "similaridade"],
    )


for categoria, termo in termos_atividade.items():
    print(f"\nTermos mais similares no Word2Vec pre-treinado - {categoria}: {termo}")
    display(similares_word2vec_pretreinado(termo, topn=10))

In [ ]:
def analogia_gensim(modelo_vetores, positivos, negativos=None, topn=10, nome_modelo="modelo"):
    negativos = negativos or []
    termos = positivos + negativos
    ausentes = [termo for termo in termos if termo not in modelo_vetores]
    if ausentes:
        return pd.DataFrame({"aviso": [f"Termos fora do vocabulario em {nome_modelo}: {', '.join(ausentes)}"]})
    return pd.DataFrame(
        modelo_vetores.most_similar(positive=positivos, negative=negativos, topn=topn),
        columns=["palavra", "similaridade"],
    )


operacoes_vetoriais = [
    {
        "descricao": "modelos + linguagem - corpus",
        "positivos": ["modelos", "linguagem"],
        "negativos": ["corpus"],
    },
    {
        "descricao": "substantivo + linguagem - verbo",
        "positivos": [substantivo_mais_frequente, "linguagem"],
        "negativos": [verbo_mais_frequente],
    },
]

for operacao in operacoes_vetoriais:
    positivos = [termo for termo in operacao["positivos"] if termo]
    negativos = [termo for termo in operacao["negativos"] if termo]

    print(f"\nOperacao vetorial: {operacao['descricao']}")
    print("Word2Vec corpus")
    display(analogia_gensim(
        word2vec_model.wv,
        positivos=positivos,
        negativos=negativos,
        topn=10,
        nome_modelo="Word2Vec corpus",
    ))

    print("Word2Vec pre-treinado")
    if word2vec_pretreinado is None:
        display(pd.DataFrame({"aviso": ["Modelo pre-treinado nao carregado."]}))
    else:
        display(analogia_gensim(
            word2vec_pretreinado,
            positivos=positivos,
            negativos=negativos,
            topn=10,
            nome_modelo="Word2Vec pre-treinado",
        ))

## Treinar GloVe simples no corpus

In [ ]:
def construir_coocorrencias(sentencas, vocab, window=5):
    cooc = defaultdict(float)
    token_to_id = {token: i for i, token in enumerate(vocab)}
    for tokens in sentencas:
        ids = [token_to_id[t] for t in tokens if t in token_to_id]
        for i, alvo in enumerate(ids):
            inicio = max(0, i - window)
            fim = min(len(ids), i + window + 1)
            for j in range(inicio, fim):
                if i == j:
                    continue
                contexto = ids[j]
                distancia = abs(i - j)
                cooc[(alvo, contexto)] += 1.0 / distancia
    return cooc, token_to_id


class GloveSimples(torch.nn.Module):
    def __init__(self, vocab_size, vector_size):
        super().__init__()
        self.wi = torch.nn.Embedding(vocab_size, vector_size)
        self.wj = torch.nn.Embedding(vocab_size, vector_size)
        self.bi = torch.nn.Embedding(vocab_size, 1)
        self.bj = torch.nn.Embedding(vocab_size, 1)

    def forward(self, i, j):
        produto = (self.wi(i) * self.wj(j)).sum(dim=1)
        return produto + self.bi(i).squeeze() + self.bj(j).squeeze()


def treinar_glove(sentencas, frequencias, vector_size=100, max_vocab=5000, window=5, epochs=25, lr=0.05):
    vocab = [token for token, freq in frequencias.most_common(max_vocab) if freq >= MIN_COUNT]
    cooc, token_to_id = construir_coocorrencias(sentencas, vocab, window=window)
    pares = list(cooc.items())

    modelo = GloveSimples(len(vocab), vector_size)
    optimizer = torch.optim.Adam(modelo.parameters(), lr=lr)

    x_max = 100
    alpha = 0.75

    for epoch in range(1, epochs + 1):
        random.shuffle(pares)
        perdas = []
        for (i, j), valor in pares:
            i_tensor = torch.tensor([i], dtype=torch.long)
            j_tensor = torch.tensor([j], dtype=torch.long)
            x = torch.tensor([valor], dtype=torch.float)
            peso = torch.where(x < x_max, (x / x_max) ** alpha, torch.ones_like(x))
            alvo = torch.log(x)

            pred = modelo(i_tensor, j_tensor)
            loss = (peso * (pred - alvo) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            perdas.append(loss.item())

        if epoch == 1 or epoch % 5 == 0:
            print(f"Época {epoch:02d} | perda média: {sum(perdas) / len(perdas):.4f}")

    with torch.no_grad():
        vetores = modelo.wi.weight + modelo.wj.weight

    return {
        "vocab": vocab,
        "token_to_id": token_to_id,
        "vectors": vetores.detach().cpu(),
    }


glove_corpus = treinar_glove(sentencas, frequencias, vector_size=VECTOR_SIZE, max_vocab=3000, epochs=20)
print("GloVe vocabulário:", len(glove_corpus["vocab"]))

## Comparar palavras similares nos três modelos

In [ ]:
def similares_glove(glove, palavra, topn=10):
    token_to_id = glove["token_to_id"]
    vocab = glove["vocab"]
    vetores = glove["vectors"]

    if palavra not in token_to_id:
        return pd.DataFrame({"aviso": [f"A palavra '{palavra}' não está no vocabulário do GloVe."]})

    idx = token_to_id[palavra]
    alvo = vetores[idx]
    sims = torch.nn.functional.cosine_similarity(alvo.unsqueeze(0), vetores)
    melhores = torch.topk(sims, k=min(topn + 1, len(vocab))).indices.tolist()
    linhas = [
        {"palavra": vocab[i], "similaridade": float(sims[i])}
        for i in melhores
        if vocab[i] != palavra
    ][:topn]
    return pd.DataFrame(linhas)


def similares_gensim(modelo, palavra, topn=10):
    if palavra not in modelo.wv:
        return pd.DataFrame({"aviso": [f"A palavra '{palavra}' não está no vocabulário."]})
    return pd.DataFrame(modelo.wv.most_similar(palavra, topn=topn), columns=["palavra", "similaridade"])


palavra_teste = top_palavras.iloc[0]["palavra"]
print("Palavra de teste:", palavra_teste)

print("\nWord2Vec")
display(similares_gensim(word2vec_model, palavra_teste))

print("\nFastText")
display(similares_gensim(fasttext_model, palavra_teste))

print("\nGloVe")
display(similares_glove(glove_corpus, palavra_teste))

## Salvar modelos e resultados

In [ ]:
word2vec_model.save("word2vec_stil2023.model")
fasttext_model.save("fasttext_stil2023.model")
torch.save(glove_corpus, "glove_stil2023.pt")

print("Arquivos gerados:")
print("- word2vec_stil2023.model")
print("- fasttext_stil2023.model")
print("- glove_stil2023.pt")
print("- nuvem_palavras_stil2023.png")
print("- palavras_mais_frequentes_stil2023.csv")